In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
import drjit as dr
import mitsuba as mi
import xarray as xr
import tqdm.auto as tqdm

mi.set_variant("scalar_mono_polarized_double")

In [ ]:
rayleigh = mi.load_dict({"type": "rayleigh"})
rayleigh_polarized = mi.load_dict({"type": "rayleigh_polarized"})

In [ ]:
# Reference implementation
def rayleigh_scatter(cos_theta, rho=0.0):
    # Based on Hansen & Travis (1974) eq. (2.15)
    delta = (1.0 - rho) / (1.0 + 0.5 * rho)
    delta_prime = (1.0 - 2.0 * rho) / (1.0 - rho)

    cos_theta2 = cos_theta**2
    a = 0.75 * (cos_theta2 + 1.0)
    b = 0.75 * (cos_theta2 - 1.0)
    c = 1.5 * cos_theta

    return np.array(
        [
            [a * delta + (1.0 - delta), b * delta, 0, 0],
            [b * delta, a * delta, 0, 0],
            [0, 0, c * delta, 0],
            [0, 0, 0, c * delta * delta_prime],
        ]
    )

In [ ]:
def sph_to_dir(theta, phi):
    """Map spherical to Euclidean coordinates"""
    st, ct = dr.sincos(theta)
    sp, cp = dr.sincos(phi)
    return mi.Vector3f(cp * st, sp * st, ct)


In [ ]:
# Create a (dummy) surface interaction to use for the evaluation of the BSDF
mei = dr.zeros(mi.MediumInteraction3f)

# Specify an incident direction with 45 degrees elevation
mei.wi = sph_to_dir(dr.deg2rad(45.0), 0.0)

# Create grid in spherical coordinates and map it onto the sphere
res = 30
ntheta = res
nphi = 2*res
theta_os = np.linspace(0, dr.pi, res)
phi_os = np.linspace(0, 2 * dr.pi, 2 * res)

data_scalar = []
data_polarized = []
data_reference = []

with tqdm.tqdm(total=res * res * 2) as pbar:
    for theta_o, phi_o in itertools.product(theta_os, phi_os):
        wo = sph_to_dir(theta_o, phi_o)
        data_scalar.append(rayleigh.eval_pdf(mi.PhaseFunctionContext(None), mei, wo)[0])
        data_polarized.append(rayleigh_polarized.eval_pdf(mi.PhaseFunctionContext(None), mei, wo)[0])
        data_reference.append(rayleigh_scatter(np.dot(wo, -mei.wi)))
        pbar.update()

data_scalar = xr.DataArray(
    data=np.reshape(data_scalar, (ntheta, nphi, 4, 4)),
    dims=["theta", "phi", "m", "n"],
    coords={
        "theta": ("theta", np.rad2deg(theta_os)),
        "phi": ("phi", np.rad2deg(phi_os)),
        "m": list(range(4)),
        "n": list(range(4)),
    },
)

data_polarized = xr.DataArray(
    data=np.reshape(data_polarized, (ntheta, nphi, 4, 4)),
    dims=["theta", "phi", "m", "n"],
    coords={
        "theta": ("theta", np.rad2deg(theta_os)),
        "phi": ("phi", np.rad2deg(phi_os)),
        "m": list(range(4)),
        "n": list(range(4)),
    },
)
data_polarized

data_reference = xr.DataArray(
    data=np.reshape(data_reference, (ntheta, nphi, 4, 4)),
    dims=["theta", "phi", "m", "n"],
    coords={
        "theta": ("theta", np.rad2deg(theta_os)),
        "phi": ("phi", np.rad2deg(phi_os)),
        "m": list(range(4)),
        "n": list(range(4)),
    },
)
data_reference

In [ ]:
data_scalar.sel(m=0, n=0).plot.imshow(aspect=nphi / ntheta, size=3)

In [ ]:
fig, axs = plt.subplots(4,4, layout="constrained", figsize=(4 * 4, 4 * 2), sharex=True, sharey=True)

for irow, icol in itertools.product(range(4), range(4)):
    ax = axs[irow, icol]
    data_polarized.sel(m=irow, n=icol).plot.imshow(ax=ax, cmap="RdBu")

In [ ]:
fig, axs = plt.subplots(4,4, layout="constrained", figsize=(4 * 4, 4 * 2), sharex=True, sharey=True)

for irow, icol in itertools.product(range(4), range(4)):
    ax = axs[irow, icol]
    data_reference.sel(m=irow, n=icol).plot.imshow(ax=ax, cmap="RdBu")